In [1]:
# Importing necessary libraries

import nltk
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import re
import random
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,GradientBoostingClassifier,BaggingClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB,MultinomialNB,ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.pipeline import Pipeline
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.feature_selection import SelectFromModel
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
nltk.download('wordnet')


ModuleNotFoundError: No module named 'sklearn.utils._metadata_requests'

In [2]:
# Reading the CSV file into a DataFrame
train = pd.read_csv('/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv')

# Displaying the first few rows of the dataset for inspection
print(train.head())


  essay_id                                          full_text  score
0  663d2cf  Dear State Senator,\n\nI am arguing in favor o...      3
1  3a20bfb  In " The Challenge of Exploring Venus" The aut...      2
2  6adae64  Teachers can have a hard time telling if their...      3
3  c81ccdf  Using dirverless cars can be very dangrous in ...      3
4  9549d7f  In " The Challenge of Exploring Venus" the aut...      3


In [3]:
# Importing the ProfileReport class from the ydata_profiling library
from ydata_profiling import ProfileReport

# Generating a profile report for the train dataset
# Setting the title of the report to "Pandas Profiling Report"
profile = ProfileReport(train, title="Pandas Profiling Report")

# Displaying the generated profile report
profile


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 3/3 [00:04<00:00,  1.49s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
%%time
#Running the Preprocessing and cleaning phase

from nltk.stem import WordNetLemmatizer
import re
#Removes Punctuations
def remove_punctuations(data):
    punct_tag=re.compile(r'[^\w\s]')
    data=punct_tag.sub(r'',data)
    return data

#Removes HTML syntaxes
def remove_html(data):
    html_tag=re.compile(r'<.*?>')
    data=html_tag.sub(r'',data)
    return data

#Removes URL data
def remove_url(data):
    url_clean= re.compile(r"https://\S+|www\.\S+")
    data=url_clean.sub(r'',data)
    return data

#Removes Emojis
def remove_emoji(data):
    emoji_clean= re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    data=emoji_clean.sub(r'',data)
    url_clean= re.compile(r"https://\S+|www\.\S+")
    data=url_clean.sub(r'',data)
    return data.lower()
#Lemmatize the corpus
def lemma_traincorpus(data):
    lemmatizer=WordNetLemmatizer()
    out_data=""
    for words in data:
        out_data+= lemmatizer.lemmatize(words)
    return out_data

def tfidf(data):
    tfidfv = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), lowercase=True, max_features=150000)
    fit_data_tfidf=tfidfv.fit_transform(data)
    return fit_data_tfidf

DESCRIPTION=train['full_text']
DESCRIPTION=DESCRIPTION.apply(lambda z: remove_punctuations(z))
DESCRIPTION=DESCRIPTION.apply(lambda z: remove_html(z))
DESCRIPTION=DESCRIPTION.apply(lambda z: remove_url(z))
DESCRIPTION=DESCRIPTION.apply(lambda z: remove_emoji(z))
#DESCRIPTION=DESCRIPTION.apply(lambda z: lemma_traincorpus(z))
train['text']=DESCRIPTION
train


CPU times: user 1.36 s, sys: 12 ms, total: 1.37 s
Wall time: 1.37 s


,essay_id,full_text,score,text
0,663d2cf,"Dear State Senator,\n\nI am arguing in favor o...",3,dear state senator\n\ni am arguing in favor of...
1,3a20bfb,"In "" The Challenge of Exploring Venus"" The aut...",2,in the challenge of exploring venus the autho...
2,6adae64,Teachers can have a hard time telling if their...,3,teachers can have a hard time telling if their...
3,c81ccdf,Using dirverless cars can be very dangrous in ...,3,using dirverless cars can be very dangrous in ...
4,9549d7f,"In "" The Challenge of Exploring Venus"" the aut...",3,in the challenge of exploring venus the autho...
...,...,...,...,...
15571,88441d4,I think people should participate in the Seago...,3,i think people should participate in the seago...
15572,c0f48f0,"In the article ""Making Mona Lisa Smile,"" the a...",1,in the article making mona lisa smile the auth...
15573,919cc62,"''Are you kidding me, you actually think that ...",3,are you kidding me you actually think that fig...
15574,9f6cb41,I feel that this could be very usefull in the ...,4,i feel that this could be very usefull in the ...


In [5]:
# Converting the 'text' column to string type and assigning to X
X = train['text'].astype(str)

# Converting the 'score' column to integer type and assigning to y
y = train['score'].astype(int)

# Splitting the dataset into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,  # Ensures that the train and test sets have approximately the same proportion of each target class
    test_size=0.33,  # Setting the test set size to 33% of the original dataset
    random_state=0  # Setting a random state for reproducibility
)


In [6]:
# Creating a pipeline for the model
model = Pipeline([
    ('Vectorizer', TfidfVectorizer()),  # TF-IDF vectorizer for text feature extraction
    ('feature_selection', SelectFromModel(ExtraTreesClassifier())),  # Feature selection using ExtraTreesClassifier
    ('clf', LinearSVC())  # Linear Support Vector Classifier
])

# Fitting the model on the training data
model.fit(X_train, y_train)

# Making predictions on the train data
predictions = model.predict(X_train)
# Calculating and printing the weighted F1 score and cohen_kappa_score
f1 = f1_score(y_train, predictions, average='weighted')  # 'weighted' or 'macro', depending on your needs
print(f'F1 score: {f1}')
score = cohen_kappa_score(y_train, predictions)  # 'weighted' or 'macro', depending on your needs
print(f'cohen kappa score: {score}')


NameError: name 'SelectFromModel' is not defined

In [7]:
# Making predictions on the test data
predictions = model.predict(X_test)

# Calculating the confusion matrix
cm = confusion_matrix(y_test, predictions, labels=model.classes_)

# Displaying the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=model.classes_)
disp.plot()
plt.show()

# Calculating and printing the weighted F1 score and cohen_kappa_score
f1 = f1_score(y_test, predictions, average='weighted')  # 'weighted' or 'macro', depending on your needs
print(f'F1 score: {f1}')
score = cohen_kappa_score(y_test, predictions)  # 'weighted' or 'macro', depending on your needs
print(f'cohen kappa score: {score}')


NameError: name 'model' is not defined

In [8]:
# Remove the classifier from the original pipeline
model.steps.pop()

# Create a new pipeline using the remaining steps
new_pipeline = Pipeline(model.steps)


NameError: name 'model' is not defined

In [9]:
import lightgbm as lgb

params = {'n_estimators': 195,
 'num_leaves': 6,
 'min_child_samples': 3,
 'learning_rate': 0.06518970520093895,
 'max_bin': 511,
 'colsample_bytree': 0.9699010403795221,
 'reg_alpha': 0.028218191448367104,
 'reg_lambda': 0.855160569748305}


X_train_vector=new_pipeline.transform(X_train)
X_test_vector=new_pipeline.transform(X_test)
lgb_model = lgb.LGBMClassifier(**params)
lgb_model.fit(X_train_vector, y_train)

# Making predictions on the train data
predictions = lgb_model.predict(X_train_vector)
# Calculating and printing the weighted F1 score and cohen_kappa_score
f1 = f1_score(y_train, predictions, average='weighted')  # 'weighted' or 'macro', depending on your needs
print(f'F1 score: {f1}')
score = cohen_kappa_score(y_train, predictions)  # 'weighted' or 'macro', depending on your needs
print(f'cohen kappa score: {score}')


# Making predictions on the test data
predictions = lgb_model.predict(X_test_vector)

# Calculating the confusion matrix
cm = confusion_matrix(y_test, predictions, labels=[x for x in range(1,7)])

# Displaying the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=[x for x in range(1,7)])
disp.plot()
plt.show()

# Calculating and printing the weighted F1 score and cohen_kappa_score
f1 = f1_score(y_test, predictions, average='weighted')  # 'weighted' or 'macro', depending on your needs
print(f'F1 score: {f1}')
score = cohen_kappa_score(y_test, predictions)  # 'weighted' or 'macro', depending on your needs
print(f'cohen kappa score: {score}')


NameError: name 'new_pipeline' is not defined

In [10]:
"""
# Define the number of splits for cross-validation
n_splits = 5

# Initialize StratifiedKFold with the specified number of splits
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

# Lists to store scores
f1_scores = []
kappa_scores = []
models = []
predictions = []
# Loop through each fold of the cross-validation
for train_index, test_index in skf.split(X, y):
    # Split the data into training and testing sets for this fold
    X_train_fold, X_test_fold = X[train_index], X[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]
    
    # Fit the model on the training data for this fold
    
    model = Pipeline([
    ('Vectorizer', TfidfVectorizer()),  # TF-IDF vectorizer for text feature extraction
    ('feature_selection', SelectFromModel(ExtraTreesClassifier())),  # Feature selection using ExtraTreesClassifier
    ('clf', LinearSVC())  # Linear Support Vector Classifier
                    ])
    model.fit(X_train_fold, y_train_fold)
    models.append(model)
    # Make predictions on the test data for this fold
    predictions_fold = model.predict(X_test_fold)
    predictions.append(predictions_fold)
    # Calculate and store the F1 score for this fold
    f1_fold = f1_score(y_test_fold, predictions_fold, average='weighted')
    f1_scores.append(f1_fold)
    
    # Calculate and store the Cohen's kappa score for this fold
    kappa_fold = cohen_kappa_score(y_test_fold, predictions_fold)
    kappa_scores.append(kappa_fold)

    
    
# Calculate the mean scores across all folds
mean_f1_score = np.mean(f1_scores)
mean_kappa_score = np.mean(kappa_scores)

# Print the mean scores
print(f'Mean F1 score across {n_splits} folds: {mean_f1_score}')
print(f'Mean Cohen kappa score across {n_splits} folds: {mean_kappa_score}')
"""


"\n# Define the number of splits for cross-validation\nn_splits = 5\n\n# Initialize StratifiedKFold with the specified number of splits\nskf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)\n\n# Lists to store scores\nf1_scores = []\nkappa_scores = []\nmodels = []\npredictions = []\n# Loop through each fold of the cross-validation\nfor train_index, test_index in skf.split(X, y):\n    # Split the data into training and testing sets for this fold\n    X_train_fold, X_test_fold = X[train_index], X[test_index]\n    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]\n    \n    # Fit the model on the training data for this fold\n    \n    model = Pipeline([\n    ('Vectorizer', TfidfVectorizer()),  # TF-IDF vectorizer for text feature extraction\n    ('feature_selection', SelectFromModel(ExtraTreesClassifier())),  # Feature selection using ExtraTreesClassifier\n    ('clf', LinearSVC())  # Linear Support Vector Classifier\n                    ])\n    model.fi

In [11]:
test=pd.read_csv('/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv')
DESCRIPTION=test['full_text']
DESCRIPTION=DESCRIPTION.apply(lambda z: remove_punctuations(z))
DESCRIPTION=DESCRIPTION.apply(lambda z: remove_html(z))
DESCRIPTION=DESCRIPTION.apply(lambda z: remove_url(z))
DESCRIPTION=DESCRIPTION.apply(lambda z: remove_emoji(z))
#DESCRIPTION=DESCRIPTION.apply(lambda z: lemma_traincorpus(z))
test['text']=DESCRIPTION
test


,essay_id,full_text,text
0,d550b2d,The face was not created by aliens because the...,the face was not created by aliens because the...
1,0c10954,Hello my name is Luke Bomberger and I was seag...,hello my name is luke bomberger and i was seag...
2,ef04816,The technology to read the emotional expressio...,the technology to read the emotional expressio...
3,88ab8d9,Do you ever wonder how many soomething is or h...,do you ever wonder how many soomething is or h...
4,cee8c0f,This story is about discovering mars. Witch is...,this story is about discovering mars witch is ...
...,...,...,...
1726,d8c9e3d,The devlopment of driverless cars should not b...,the devlopment of driverless cars should not b...
1727,6c3e2b9,this story is about how hard it is about how h...,this story is about how hard it is about how h...
1728,743f56b,Do you think the author did a great job on sug...,do you think the author did a great job on sug...
1729,03630c7,Have you ever heard of the Seagoing Cowboys pr...,have you ever heard of the seagoing cowboys pr...


In [12]:
test_vector=new_pipeline.transform(test['text'])
predictions=lgb_model.predict(test_vector)
predictions


NameError: name 'new_pipeline' is not defined

In [13]:
#predictions = model.predict(test['text'])
#predictions
# Initialize an empty list to store probabilities for each model
"""
probabilities = []

# Iterate over each model in the list of models
for model in models:
    # Make predictions using the current model
    print(model.predict(test['text']))
    
    # Compute decision scores for the current model
    decision_scores = model.decision_function(test['text'])
    
    # Convert decision scores to probabilities using the sigmoid function
    proba = 1 / (1 + np.exp(-decision_scores))
    # Append the probabilities to the list
    probabilities.append(proba)

# Compute the average probabilities across all models
probabilities = np.mean(probabilities, axis=0)

# Determine the predicted class by selecting the class with the highest probability
# Add 1 to the index to match the class labels (assuming classes start from 1)
predictions = np.argmax(probabilities, axis=1) + 1

# Print the predictions
print(predictions)
"""


"\nprobabilities = []\n\n# Iterate over each model in the list of models\nfor model in models:\n    # Make predictions using the current model\n    print(model.predict(test['text']))\n    \n    # Compute decision scores for the current model\n    decision_scores = model.decision_function(test['text'])\n    \n    # Convert decision scores to probabilities using the sigmoid function\n    proba = 1 / (1 + np.exp(-decision_scores))\n    # Append the probabilities to the list\n    probabilities.append(proba)\n\n# Compute the average probabilities across all models\nprobabilities = np.mean(probabilities, axis=0)\n\n# Determine the predicted class by selecting the class with the highest probability\n# Add 1 to the index to match the class labels (assuming classes start from 1)\npredictions = np.argmax(probabilities, axis=1) + 1\n\n# Print the predictions\nprint(predictions)\n"

In [14]:
submission=pd.read_csv("/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv")
submission['score']=predictions
submission.to_csv("submission.csv",index=None)
display(submission.head())


NameError: name 'predictions' is not defined